# 自定义中间件-Node-style hooks

## 1.基本用法

## 1.1基于装饰器的实现

In [ ]:
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_model, after_model, before_agent, after_agent
from typing import Any

@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_model <----"
    return None   # 不干扰主进程

@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_model <----"
    return None


@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_agent <----"
    return None


@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_agent <----"
    return None

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)

In [ ]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model = model,
    middleware = [
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware
    ]
)

res = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in res["messages"]:
    msg.pretty_print()